# SEC 原件台积电季度收入指引增长的 Qlib 来源映射

只读已备份 64 个季度原件，统一原公告币种/汇率并从 EDGAR 接收日期的下一 Qlib 会话开始供给 Stage1 信号。

In [1]:
import os,json,hashlib,re,tarfile,tempfile,subprocess,shlex
from pathlib import Path
from collections import Counter
from bs4 import BeautifulSoup
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from src.data.nas_market_data import _settings,archive_nas_recorder
root=Path.cwd();card=root/'research/stage1_tsmc_sec_guidance_growth_source_20261023';plan=json.loads((card/'plan.json').read_text());parent=root/plan['parent']['card'];raw=root/'data/snapshots/stage1_tsmc_sec_6k_guidance_source_probe_20261023'
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists() and not (card/'qlib_recorder').exists()
for name,key in [('quarterly_inventory_result.json','quarterly_inventory_result_sha256'),('submissions_index_result.json','submissions_index_result_sha256'),('acceptance_clock_result.json','acceptance_clock_result_sha256')]:assert sha(parent/name)==plan['parent'][key]
source=json.loads((parent/'quarterly_inventory_result.json').read_text());assert source['status']=='SEC_TSMC_quarterly_guidance_2011_2026_all_64_months_identified' and source['NAS_archive_sha256']==plan['parent']['raw_archive_sha256'] and source['NAS_remote_and_scoped_restore_verified']
state={'status':'started','plan_sha256':sha(card/'plan.json'),'model_fits':0,'native_accounts':0,'market_outcome_reads':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
rows=[]
for event in source['matches']:
 p=raw/event['raw_name'];assert sha(p)==event['raw_sha256'];b=p.read_bytes();head=b[:12000].decode('utf-8',errors='replace');assert event['accession'] in head and re.search(r'CENTRAL INDEX KEY:\s*0*1046179',head)
 accepted=re.search(r'ACCEPTANCE-DATETIME>\s*(\d{14})',head);assert accepted and accepted.group(1)==event['accepted_header'] and accepted.group(1)[:8]==event['filed'].replace('-','')
 parts=[]
 for doc in re.findall(rb'<DOCUMENT>(.*?)</DOCUMENT>',b,re.S|re.I):
  typ=re.search(rb'<TYPE>\s*([^\r\n<]+)',doc,re.I);content=re.search(rb'<TEXT>(.*?)</TEXT>',doc,re.S|re.I)
  if typ and content and typ.group(1).strip().upper() in (b'6-K',b'EX-99.1'):
   parts.append(BeautifulSoup(content.group(1).decode('utf-8',errors='replace'),'html.parser').get_text(' ',strip=True))
 text=re.sub(r'\s+',' ',' '.join(parts))
 title=re.search(r'TSMC\s+Reports\s+(First|Second|Third|Fourth)\s+Quarter\s+EPS',text,re.I);assert title and title.group(1).lower()==event['reported_quarter_ordinal']
 actual=re.findall(r'consolidated revenue of\s+NT\$\s*([\d,.]+)\s*billion',text,re.I);assert actual and len(set(actual))==1,(event['reporting_month'],actual)
 forecast=re.findall(r'Revenue\s+(?:is\s+expected\s+to\s+be|to\s+be)\s+between\s+(US\$|NT\$)\s*([\d,.]+)\s*billion\s+and\s+(US\$|NT\$)\s*([\d,.]+)\s*billion',text,re.I);assert forecast,(event['reporting_month'],'no forecast')
 currency,low_s,currency_hi,high_s=forecast[0];assert currency==currency_hi
 low=float(low_s.replace(',',''));high=float(high_s.replace(',',''));reported=float(actual[0].replace(',',''));assert 0<low<high and reported>0
 if currency=='US$':
  fx_matches=re.findall(r'exchange rate assumption of\s+1\s+US\s+dollar\s+to\s+([\d,.]+)\s+NT\s+dollars',text,re.I);assert fx_matches and len(set(fx_matches))==1,(event['reporting_month'],fx_matches)
  fx=float(fx_matches[0].replace(',',''));assert 15<fx<50
 else:fx=1.0
 midpoint=(low+high)/2;mid_nt=midpoint*fx;growth=mid_nt/reported-1;assert np.isfinite(growth) and -1<growth<2
 rows.append({'reporting_month':event['reporting_month'],'accession':event['accession'],'filed_date':pd.Timestamp(event['filed']),'accepted_header':event['accepted_header'],'raw_sha256':event['raw_sha256'],'reported_current_revenue_NT_billion':reported,'next_guidance_currency':currency,'guidance_low_billion_native_currency':low,'guidance_high_billion_native_currency':high,'issuer_assumed_NT_per_USD':fx if currency=='US$' else np.nan,'next_guidance_mid_NT_billion':mid_nt,'TSMC_next_guidance_mid_NT_over_current_revenue_minus1':growth})
events=pd.DataFrame(rows).sort_values('filed_date').reset_index(drop=True);assert len(events)==64 and events.reporting_month.is_unique and events.filed_date.is_monotonic_increasing
counts=Counter(events.next_guidance_currency);assert counts=={'NT$':26,'US$':38}
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time='2011-01-03',end_time='2026-10-09',freq='day'));assert calendar.is_unique and calendar.is_monotonic_increasing and len(calendar)==3966
available=[]
for d in events.filed_date:
 later=calendar[calendar>d];available.append(later[0] if len(later) else pd.NaT)
events['first_eligible_signal_date']=pd.to_datetime(available)
assert events.first_eligible_signal_date.dropna().is_monotonic_increasing and events.first_eligible_signal_date.notna().sum()==63
source_rows=events.loc[events.first_eligible_signal_date.notna(),['first_eligible_signal_date','reporting_month','accession','filed_date','TSMC_next_guidance_mid_NT_over_current_revenue_minus1']].sort_values('first_eligible_signal_date')
left=pd.DataFrame({'signal_date':calendar});mapped=pd.merge_asof(left,source_rows,left_on='signal_date',right_on='first_eligible_signal_date',direction='backward',allow_exact_matches=True);mapped.index=calendar;mapped['source_known']=mapped['TSMC_next_guidance_mid_NT_over_current_revenue_minus1'].notna()
assert mapped.loc[mapped.source_known,'signal_date'].gt(mapped.loc[mapped.source_known,'filed_date']).all() and mapped.loc[~mapped.source_known,'TSMC_next_guidance_mid_NT_over_current_revenue_minus1'].isna().all()
train=mapped.loc['2011-01-03':'2022-12-30'];held=mapped.loc['2023-01-03':'2026-10-07'];assert len(train)==3020 and len(held)==944
static=mapped[['TSMC_next_guidance_mid_NT_over_current_revenue_minus1','source_known']].copy();static.index=pd.MultiIndex.from_arrays([calendar,['SOXL_SOXX_CASH']*len(calendar)],names=['datetime','instrument']);static=pd.concat({'feature':static[['TSMC_next_guidance_mid_NT_over_current_revenue_minus1']],'meta':static[['source_known']]},axis=1)
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':('2011-01-03','2022-12-30'),'held':('2023-01-03','2026-10-07')});assert len(dataset.prepare('train',col_set=['feature','meta']))==3020 and len(dataset.prepare('held',col_set=['feature','meta']))==944;dataset.config(dump_all=True,recursive=True)
audit={'status':'original_SEC_TSMC_guidance_growth_64_events_Qlib_calendar_mapped_development_only','plan_sha256':sha(card/'plan.json'),'event_rows':len(events),'event_currency_counts':dict(counts),'events_within_Stage1_provider':int(events.first_eligible_signal_date.notna().sum()),'train_calendar_rows':len(train),'held_calendar_rows':len(held),'known_train_sessions':int(train.source_known.sum()),'known_held_sessions':int(held.source_known.sum()),'unknown_train_sessions':int((~train.source_known).sum()),'unknown_held_sessions':int((~held.source_known).sum()),'feature_min':float(events['TSMC_next_guidance_mid_NT_over_current_revenue_minus1'].min()),'feature_max':float(events['TSMC_next_guidance_mid_NT_over_current_revenue_minus1'].max()),'SEC_API_Z_timestamp_used_as_clock':False,'issuer_first_public_exact_clock_qualified':False,'model_fits':0,'native_accounts':0,'market_outcome_reads':0};(card/'source_audit.json').write_text(json.dumps(audit,indent=2)+'\n')
tracking=card/'qlib_recorder'
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_result_hashes=plan['parent']);rec.save_objects(artifact_path='sources',events=events);rec.save_objects(artifact_path='analysis',mapped_signals=mapped,source_audit=audit);rec.save_objects(artifact_path='metadata',dataset=dataset);rec.log_metrics(source_event_rows=64,known_train_days=audit['known_train_sessions'],known_held_days=audit['known_held_sessions'],model_fits=0,native_accounts=0);rid=rec.id;eid=str(rec.experiment_id)
with R.uri_context(str(tracking)):
 rec=R.get_recorder(recorder_id=rid,experiment_id=eid);assert len(rec.load_object('sources/events'))==64 and len(rec.load_object('analysis/mapped_signals'))==3966 and len(rec.load_object('metadata/dataset').prepare('held',col_set=['feature','meta']))==944
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
with tempfile.TemporaryDirectory(prefix='tsmc-guidance-source-restore-') as td:
 archive=Path(td)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=120)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as z:names=[m.name for m in z.getmembers() if m.isfile()]
 assert any(x.endswith('/artifacts/sources/events') for x in names) and any(x.endswith('/artifacts/analysis/mapped_signals') for x in names) and any(x.endswith('/artifacts/metadata/dataset') for x in names)
summary={'experiment':plan['experiment'],'decision':audit['status'],'objective_achieved':False,'plan':'plan.json','source_audit':'source_audit.json','recorder_id':rid,'experiment_id':eid,'recorder_uri':str(tracking),'nas_evidence':evidence,'nas_readback':{'sha256':evidence['sha256'],'verified':True},'model_fits':0,'native_accounts':0,'artifact_policy':plan['artifact_policy']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');state.update(status='completed',recorder_id=rid,nas_archive_sha256=evidence['sha256']);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');print(audit);print({'recorder_id':rid,'NAS':evidence['sha256']})


{'status': 'original_SEC_TSMC_guidance_growth_64_events_Qlib_calendar_mapped_development_only', 'plan_sha256': '3d94e00fc1ff77e3b8e94802d80a941ae6b9a286c4842e9679c0501b3c33cbc0', 'event_rows': 64, 'event_currency_counts': {'NT$': 26, 'US$': 38}, 'events_within_Stage1_provider': 63, 'train_calendar_rows': 3020, 'held_calendar_rows': 944, 'known_train_sessions': 3002, 'known_held_sessions': 944, 'unknown_train_sessions': 18, 'unknown_held_sessions': 0, 'feature_min': -0.2187597059737032, 'feature_max': 0.22453110241532848, 'SEC_API_Z_timestamp_used_as_clock': False, 'issuer_first_public_exact_clock_qualified': False, 'model_fits': 0, 'native_accounts': 0, 'market_outcome_reads': 0}
{'recorder_id': '9770e4b8e19c488f95221fc4a8150f11', 'NAS': 'a5c03bdb88a450e0244330d375c8ab51fe5bc05b104851066ae1befeb6c25f03'}


## 中文结论

64 期 SEC 原始指引数值统一币种与汇率后，Qlib 全日历训练/复用评价已知 3002/944 日，Recorder 与 NAS 回读通过。首发精确时钟尚未合格，零拟合、零账户。详见 `conclusion.md`。